## 0. 환경 구성

### 1) 라이브러리 설치

In [ ]:
# uv add langchain_community faiss-cpu langchain-ollama

In [ ]:
# ollama run qwen2.5:0.5b

# ollama pull bge-m3

In [15]:
EMBEDDING_MODEL_NAME = "bge-m3:latest"
print(EMBEDDING_MODEL_NAME)

CHAT_MODEL_NAME = "qwen2.5:0.5b"
print(CHAT_MODEL_NAME)

# FAISS 인덱스(벡터 DB) 저장 경로
# 경로 구분자는 슬래시(/)를 사용: "..\db\faiss..." 처럼 역슬래시를 쓰면 \f 가 이스케이프 문자로 해석되어 오류 발생
DB_PATH = "../db/faiss_index_ollama"
print(DB_PATH)

# version1(개선 버전) 인덱스 저장 경로: 기본 인덱스(DB_PATH)를 덮어쓰지 않도록 별도 경로 사용
DB_PATH_V1 = "../db/faiss_index_ollama_v1"
print(DB_PATH_V1)

bge-m3:latest
qwen2.5:0.5b
../db/faiss_index_ollama


#### RAG 파이프 라인
* Load Data - Text Split - Indexing - Retrieval - Generation
* OllamaEmbeddings 사용

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_ollama import ChatOllama
from langchain_ollama import OllamaEmbeddings

from pprint import pprint

# 1. Load Data
loader = TextLoader("../data/taxinfo.txt", encoding="utf-8")
documents = loader.load()

print(type(documents), len(documents)) #[Document, Document]
print(type(documents[0]))
print(documents[0])

In [ ]:

# 2️. Text Split
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_docs = splitter.split_documents(documents)

print(len(split_docs), type(split_docs), type(split_docs[0]))
print(split_docs[0])
print('두번째 Document ====================')
print(split_docs[1])


In [ ]:

# 3️. Indexing (벡터 저장)
embeddings_model = OllamaEmbeddings(model=EMBEDDING_MODEL_NAME)
# vector DB에 저장
vectorstore = FAISS.from_documents(
    documents=split_docs, 
    embedding=embeddings_model
)

# 로컬 파일로 저장
vectorstore.save_local(DB_PATH)
print(f"✓ FAISS 인덱스 저장 완료: {DB_PATH}")

In [ ]:

# 4️. Retrieval (유사 문서 검색) k: 질의와 가장 유사한 문서(청크) 6개를 찾아 반환하기
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})
# retriever변수는 VectoreStoreRetriever 객체이다.
print(type(retriever))
# **질문(쿼리)**에 대해 유사한 문서를 검색하는 역할
retrieved_docs = retriever.invoke("소득세법에서 비과세소득에 해당하는 소득은 무엇인가요?")
print(type(retrieved_docs), len(retrieved_docs))
print(type(retrieved_docs[0]))
print(retrieved_docs[0])

In [ ]:

# 5️. Generation (LLM 응답 생성)
# temperature=0.1 설정으로 되돌려도 오류는 사라집니다. 다만 실행마다 답변이 달라져 공정한 비교에는 불리합니다.
# qwen2.5:1.5b 이상의 모델이 반복에 더 강합니다.
try:
    llm = ChatOllama(
        model=CHAT_MODEL_NAME,
        temperature=0,  # 같은 입력이면 같은 결과가 나오도록 (비교 실험의 재현성)
        num_predict=1500,  # 답변이 중간에 잘리지 않도록 충분히 크게 설정
        repeat_penalty=1.2  # 같은 토큰 반복 억제 (없으면 소형 모델이 temperature=0 에서 반복 루프에 빠져 오류 발생)
    )
    print("LLM 설정 완료")
except Exception as e:
    print(f"LLM 설정 실패: {e}")
    exit(1)
    
# 유사도 검색의 결과를 str 문자열로 만들기
context = "\n\n".join([doc.page_content for doc in retrieved_docs])
print(context)

#### context 유무 비교 (RAG 효과 확인)

같은 질문을 **검색된 context를 주었을 때**와 **주지 않았을 때**로 나누어 답변을 비교합니다. 공정한 비교가 되려면 **context 유무만 다르고** 나머지 조건은 같아야 합니다.

| 조건 | 두 실험에서 |
|---|---|
| 모델, 생성 설정(`llm`) | 같음 (`llm` 하나를 공유, `temperature=0`) |
| 질문(`QUESTION`) | 같음 |
| 프롬프트 형식 | 같은 함수 `ask()`를 사용, `[관련 정보]` 부분만 있고 없고 |
| **context** | **다름 (이 차이만 비교)** |

> 이 노트북의 `qwen2.5` 소형 모델은 매우 작아서 context를 주어도 일부 내용을 빠뜨리거나 잘못 옮길 수 있습니다. 두 결과를 `data/taxinfo.txt`의 **제12조 원문과 비교**하여, 어느 답변이 원문에 충실한지 직접 확인하세요.

In [24]:
QUESTION = "소득세법에서 비과세소득에 해당하는 소득은 무엇인가요?"


def ask(question: str, context: str | None = None) -> str:
    """context 유무만 다르고 나머지 조건(모델, 질문)은 같은 질의 함수"""
    if context:
        prompt = f"다음 관련 정보를 근거로 질문에 답하세요.\n\n[관련 정보]\n{context}\n\n[질문]\n{question}"
    else:
        prompt = f"질문에 답하세요.\n\n[질문]\n{question}"
    return llm.invoke(prompt).content

In [25]:
print('context 적용한 결과')
pprint(ask(QUESTION, context))

context 적용한 결과
("소득세법에서는 공익신탁, 사업소득, 근로소득, 퇴직소득 등 4개의 소득을 제외하고는 비과세소득이 있습니다. 이 중 '공익신타피'와 "
 "'사업소득', '근로소득', '퇴직소득'에 해당하는 경우를 고려해 볼 수 있습니다.\n"
 '\n'
 '1. **공익신탰피**:\n'
 '   - 논밭을 작물 생산에 이용하게 함으로써 발생하는 소득\n'
 '     - 2013년부터 제한된 범위에서 공익실타피는 과세하지 않습니다.\n'
 '   \n'
 '2. **사업소득**:\n'
 '   - 99조와 같은 규정 아래로 분류됩니다.\n'
 '\n'
 '3. **근로소득과 퇴직소득**:\n'
 '   - 대통령령으로 정하는 농어가부업소득, 전통주의 제조에서 발생하는 소득 등에 해당합니다.\n'
 '   \n'
 '4. **퇴직소득**:\n'
 '   - 2018년 3월 20일부터 2023년 8월 8일까지의 범위를 초과한 경우에, 대통령령으로 정하는 농어가부업소득 및 전통주의 '
 '제조에서 발생하는 소득을 포함합니다.\n'
 '\n'
 '5. ** 기타소득**:\n'
 "   - '국가유공자 등 예우 및 지원에 관한 법률' 또는 '보훈보상대상자 지원에 관한 법률', '북한이탈주민의 보호 및 정착지원에 관한 "
 "법률'등에 따라 받는 금액을 포함합니다.\n"
 '\n'
 "따라서, 소득세법에서 비과세소득은 공익신탰피와 사업소득, 근로소득, 퇴직소득 등 4개의 소득입니다. 이 중 '공익실타피'는 과세하지 "
 '않습니다.')


In [26]:
print('context 적용하지 않은 결과')
pprint(ask(QUESTION))

context 적용하지 않은 결과
('소득세법에서는 "비과세 소득"이란 특정 범위의 소득을 제외하고, 그 중 일부를 대상으로 세금을 부과합니다. 이는 일반적으로 기업이나 개인 '
 '등에서 발생하는 비용 또는 수익 등을 포함한 모든 소득에 대한 정책입니다.\n'
 '\n'
 '소득세법에서는 "비과세 소득"은 다음과 같은 특징이 있습니다:\n'
 '\n'
 '1. **부인의 범위**: \n'
 '   - 부인이 자녀 2명 이상일 경우, 그들의 합계를 제외합니다.\n'
 '   \n'
 '2. **대상 범위**:\n'
 '   - 기업: 전년도 실적에 따라\n'
 '   - 개인: 연간 소득을 초과한 경우\n'
 '\n'
 '3. **세금 대비 비용**: \n'
 '   - 부인의 자녀 1명 이상일 경우, 그들의 합계를 제외합니다.\n'
 '   \n'
 '4. **소득 세율**:\n'
 '   - 기업: 전년도 실적에 따라\n'
 '   - 개인: 연간 소득을 초과한 경우\n'
 '\n'
 '5. **세금 대비 비용**: \n'
 '   - 부인의 자녀 1명 이상일 경우, 그들의 합계를 제외합니다.\n'
 '\n'
 '이러한 특징들은 각각 기업이나 개인에게 적용되는 세율에 따라 달라집니다. 따라서 소득세법에서 "비과세소득"은 특정 범위 내에서 발생하는 '
 '비용 또는 수익을 대상으로 세금을 부과할 때 사용됩니다.\n'
 '\n'
 '이 정보는 일반적인 사례이며, 실제 구체적 적용 방식에 따라 달라질 수 있습니다.')


### 개선한 Source - version1
* Retriever 검색방법 개선
    * search_type="mmr",  # 최대 다양성 검색
    * search_kwargs={"k": 3, "fetch_k": 6}  

In [ ]:

from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_ollama import OllamaEmbeddings

from pprint import pprint

# 1. 데이터 로드 (기존과 동일)
loader = TextLoader("../data/taxinfo.txt", encoding="utf-8")
documents = loader.load()

# 2. 텍스트 분할 개선
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # 크기 증가
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""],  # 자연스러운 분할을 위한 구분자
    length_function=len,
    is_separator_regex=False,
)
split_docs = splitter.split_documents(documents)

# 3. 인덱싱 (벡터 저장)
embeddings_model = OllamaEmbeddings(model=EMBEDDING_MODEL_NAME)

vectorstore = FAISS.from_documents(
    documents=split_docs, 
    embedding=embeddings_model
)
#vectorstore = FAISS.from_documents(split_docs, OpenAIEmbeddings())

vectorstore.save_local(DB_PATH_V1)
print(f"✓ FAISS 인덱스 저장 완료: {DB_PATH_V1}")

# 4. 검색 개선
"""
    최대 다양성 검색(Maximum Marginal Relevance, MMR)
    MMR은 유사도가 높은 문서를 찾는 것을 넘어, 유사도와 다양성이라는 두 가지 기준을 모두 고려함
    - search_type="mmr": 검색 방식을 MMR로 지정합니다.
    - fetch_k: 일차적으로 질의와 유사한 문서 6개를 벡터 저장소에서 가져옵니다.
    - k: fetch_k로 가져온 6개의 문서 중에서 최종적으로 3개를 선택합니다. 3개를 선택할 때, MMR 알고리즘은 다음 두 가지를 고려함
        : 질의와의 유사도가 높고, 이미 선택된 다른 문서들과의 유사도가 낮은 (즉, 내용이 다양한) 문서
    * MMR의 작동 원리:
    - 질의와 가장 유사한 fetch_k개(6개)의 문서를 예비 후보군으로 가져옵니다.
    - 이 후보군 중에서 질의와 가장 유사한 문서 하나를 첫 번째 결과로 선택합니다.
    - 남은 후보군 중에서 질의와의 유사도는 높으면서 (관련성), 
      이미 선택된 문서들과의 유사도는 낮은 (다양성) 문서를 찾아 다음 결과로 추가함
    - 이 과정을 k개(3개)의 문서가 모두 선택될 때까지 반복합니다.
    * 주의: 이 문서는 청크가 7개뿐이라 k=6, fetch_k=10 이면 거의 모든 청크(원문의 98%)가 선택되어 검색의 의미가 없어짐        
"""
retriever = vectorstore.as_retriever(
    search_type="mmr",  # 최대 다양성 검색
    search_kwargs={"k": 3, "fetch_k": 6}  # 전체 청크(7개)보다 작게 설정
)

# 5. 프롬프트 엔지니어링
def generate_prompt(query, context):
    return f"""다음은 소득세법 비과세소득 관련 조항입니다. 문맥을 고려하여 질문에 답변하세요.

[관련 조항]
{context}

[질문]
{query}

[답변 요구사항]
- 비과세소득 유형을 계층적으로 구분하여 설명
- 각 항목별 구체적인 조건 명시
- 법조문의 항, 호, 목 번호를 포함
- 500자 이내로 간결하게 요약"""

# 검색 및 응답 생성
query = "소득세법에서 비과세소득에 해당하는 소득은 무엇인가요?"
retrieved_docs = retriever.invoke(query)
context = "\n\n".join([doc.page_content for doc in retrieved_docs])

# LLM 은 위(5. Generation)에서 만든 llm 을 재사용
# (temperature=0, num_predict=1500, repeat_penalty=1.2 → 앞 비교와 같은 조건)
response = llm.invoke(generate_prompt(query, context))

print('개선된 결과:')
pprint(response.content)